1. What is Text Splitting?
Text splitting is the process of breaking down massive chunks of text (like articles, PDFs, HTML pages, or books) into smaller, more manageable pieces
. The tool or code that performs this operation is called a Text Splitter

2. Why is Text Splitting Necessary?
There are three primary reasons to perform text splitting when building LLM applications:
Overcoming Model Limitations (Context Length): Every LLM has a strict limit on how much text it can process at once (its context window, e.g., 50,000 tokens)
. If a PDF has over 100,000 words, it will breach this threshold; chunking allows the LLM to process it
.
Improving Downstream Tasks:
Embeddings: A massive text chunk generates poor-quality embedding vectors that fail to accurately capture semantic meaning
. Generating separate embeddings for smaller, focused chunks (e.g., separate paragraphs for different IPL teams) yields much higher quality representations
.
Semantic Search & Summarization: Chunking results in more precise search results and reduces the chances of an LLM drifting off-topic or hallucinating when summarizing

3. Crucial Concept: Chunk Overlap
When splitting text, you risk cutting off a sentence or paragraph mid-way, losing vital context between the end of one chunk and the start of the next
. Chunk Overlap solves this by keeping a specific number of characters shared between two consecutive chunks
. This acts as a bridge, ensuring the context is passed on
. As a rule of thumb for RAG applications, setting the chunk overlap to 10% to 20% of the total chunk size is considered optimal

4. Types of Text Splitters (with Code)
A. Length-Based Text Splitting (CharacterTextSplitter)
This is the simplest and fastest method. It strictly divides the text based on a pre-defined character or token count
.
The Drawback: It ignores grammar, linguistic structure, and meaning. Text will frequently be cut abruptly in the middle of words or sentences, leading to a loss of context

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

text = "Your massive document text goes here..."

# Initialize the Splitter
splitter = CharacterTextSplitter(
    chunk_size=100,      # Max characters per chunk
    chunk_overlap=10,    # Characters shared between chunks
    separator=""         # Split exactly at the chunk size limit
)

# Split a raw string
chunks = splitter.split_text(text)
print(chunks) 

# Note: Integrating with Document Loaders: If you are using a Document Loader (like PyPDFLoader), 
# you should use the split_documents() function instead of split_text(). 
# This takes a list of Document objects, splits them, and returns a new list of smaller Document objects


# Assuming 'docs' is a list of Document objects loaded via PyPDFLoader
split_docs = splitter.split_documents(docs)
print(split_docs[0].page_content)  # Accessing the first chunk's text




In [ ]:
# # B. Text Structure-Based Text Splitting (RecursiveCharacterTextSplitter)

# This is the most widely used and recommended text splitter
# . It respects the natural hierarchy of written language: Paragraphs (\n\n) → Sentences (\n) → Words (spaces) → Characters

# How it works: It tries to split the text by paragraphs first. If a resulting paragraph is still larger than the chunk_size, it recursively drops down to split by sentences, 
# then words, and finally characters, always trying to keep semantically linked text together

from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_text(text)




In [ ]:
# C. Document-Based Text Splitting (For Code & Markdown)

# Standard text splitters fail on non-plain text documents like Python scripts or Markdown files, because these files are organized via syntax (like class, def, or #), not paragraphs
# . LangChain extends the recursive splitter to handle these formats by using language-specific separators

from langchain.text_splitter import RecursiveCharacterTextSplitter, Language

python_code = """class MyClass: ..."""

# Initialize a language-specific splitter
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, # Can also be Language.MARKDOWN, Language.HTML, etc.
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_text(python_code)


In [ ]:
# D. Semantic Meaning-Based Text Splitting (SemanticChunker)

# Sometimes a single paragraph discusses two completely unrelated topics (e.g., farming and cricket)
# . Structure-based splitters will group them together, yielding poor embeddings
# . Semantic splitting solves this by looking at the actual meaning of the text


# How it works: It splits the text into sentences, generates an embedding vector for each sentence, and calculates the cosine similarity between consecutive sentences
# . If there is a sudden, massive drop in similarity (e.g., greater than a specified standard deviation), the algorithm assumes the topic has changed and creates a split
# .
# Note: This feature is highly promising but currently resides in LangChain's experimental library as it isn't always perfectly accurate yet

from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# Initialize the Semantic Chunker
semantic_splitter = SemanticChunker(
    OpenAIEmbeddings(), 
    breakpoint_threshold_type="standard_deviation" # Can also be percentile, interquartile, etc.
)

chunks = semantic_splitter.split_text(text)
